# ENTENDIENDO LOS DATOS
#### 1. Organiza por carpetas de 2024 y lectura


In [1]:
# Lee los archivos del 2024
import pandas as pd

mod1_2024 = pd.read_csv(
    r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\2024\Enaho01-2024-200.csv",
    encoding="latin-1"
)

# Nombres de columnas en minúscula
mod1_2024.columns = mod1_2024.columns.str.lower()

# Vistaso de los datos

print(mod1_2024.shape)
mod1_2024.head()

(117721, 40)


,año,mes,conglome,vivienda,hogar,codperso,ubigeo,dominio,estrato,p201p,...,ocupac_r3,ocupac_r4,rama_r3,rama_r4,codtarea,codtiempo,ticuest01,facpob07,nconglome,sub_conglome
0,2024,9,10651,13,11,1,150509,2,4,20240106510131101,...,,,,,,,2,115.814941,30507,0
1,2024,9,10651,13,11,2,150509,2,4,20240106510131102,...,,,,,,,2,115.814941,30507,0
2,2024,9,10651,13,11,3,150509,2,4,20240106510131103,...,,,,,,,2,115.814941,30507,0
3,2024,9,10651,13,11,4,150509,2,4,20240106510131104,...,,,,,,,2,115.814941,30507,0
4,2024,9,10651,26,11,1,150509,2,4,20240106510261101,...,,,,,,,2,115.814941,30507,0


RESPONDIENDO LAS PREGUNTAS
¿Cómo altera esto nuestro problema? ¿Cuáles son las variables que podemos
utilizar?

Esto altera bastante el problema porque cuando result no es 1 ni 2, la entrevista nunca se llegó a completar (rechazo, hogar ausente, vivienda desocupada, etc.), entonces todas las variables sustantivas del hogar (ingresos, servicios, características de la vivienda, número de miembros) quedan vacías para esas filas. Esto no es un dato faltante al azar, sino un vacío estructural, porque esa información simplemente nunca se recogió. Por eso el problema deja de ser una clasificación normal con muchas variables ricas y pasa a ser un problema de predicción de no-respuesta/cooperación del hogar. Si uso variables que solo existen cuando la entrevista sí se completó, estaría metiendo fuga de información, porque estaría usando como predictor algo que solo aparece cuando ya sé el resultado que quiero predecir. 

Observa el módulo 2: ¿podemos utilizarlo?

No podemos usarlo. Esto se debe a que el Módulo 2 solo contiene hogares donde la entrevista sí se completó, ya que no hay registro de miembros para hogares con result distinto de 1/2 (si nadie respondió, no hay roster que registrar). Entonces el Módulo 2 no cubre la clase que nos interesa predecir (la no-respuesta), y además cualquier variable suya usada como predictor implicaría fuga de información, porque solo existe porque la entrevista ya fue exitosa. Puede servir para otro problema, pero no para predecir result.


#### 2. RESULT/AÑO DEL 2019 A 2025

In [3]:
from pathlib import Path
from IPython.display import display
import pandas as pd

# ---------------------------------------------------------
# Configuración de Rutas y Variables
# ---------------------------------------------------------
BASE = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\2025")
CARPETA_SALIDA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")
AÑOS = range(2019, 2026)

VARIABLES_UTILES = [
    "conglome",
    "vivienda",
    "hogar",
    "ubigeo",
    "dominio",
    "estrato",
    "mes",
    "result",
]


# ---------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------
def leer_modulo1(base: Path, año: int) -> pd.DataFrame | None:
  archivo = base / f"Enaho01-{año}-100.csv"
  try:
    df = pd.read_csv(archivo, encoding="utf-8", sep=None, engine="python")
  except UnicodeDecodeError:
    df = pd.read_csv(archivo, encoding="latin-1", sep=None, engine="python")
  except Exception as e:
    print(f"⚠️ Error leyendo {año}: {e}")
    return None

  df.columns = df.columns.str.lower()
  return df


def filtrar_variables(df: pd.DataFrame, variables: list[str]) -> pd.DataFrame:
  cols_disponibles = [c for c in variables if c in df.columns]
  return df[cols_disponibles].copy()


# ---------------------------------------------------------
# Ejecución Principal
# ---------------------------------------------------------
def main():
  modulos_1 = {}

  # 1. Leer dataframes por año
  for año in AÑOS:
    df = leer_modulo1(BASE, año)
    if df is not None:
      modulos_1[año] = df
      print(f"{año}: {df.shape[0]} filas, {df.shape[1]} columnas")

  if not modulos_1:
    print(
        "⚠️ No se pudo cargar ningún archivo. Verifica la ruta en la variable"
        " 'BASE'."
    )
    return

  # 2. Generar el cuadro mejorado de 'result'
  mapa_result = {
      1: "01. Completa",
      2: "02. Incompleta",
      3: "03. Rechazo",
      4: "04. Ausente",
      5: "05. Deshabitada",
      6: "06. Destruida / En construcción",
      7: "07. Otro",
  }

  tabla_result = pd.DataFrame({
      año: df["result"].map(mapa_result).value_counts(normalize=True) * 100
      for año, df in modulos_1.items()
  }).fillna(0)

  tabla_result = tabla_result.sort_index()
  tabla_result.loc["Total"] = tabla_result.sum()

  # Guardar reporte en CSV
  CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
  tabla_result.round(2).to_csv(
      CARPETA_SALIDA / "porcentaje_result_por_año.csv", encoding="utf-8"
  )
  print(f"\nGuardado: {CARPETA_SALIDA / 'porcentaje_result_por_año.csv'}\n")

  # Aplicar diseño con gradiente
  estilo_result = (
      tabla_result.style.format("{:.2f}%")
      .set_caption(
          "Distribución Porcentual del Resultado de Encuesta (2019-2025)"
      )
      .background_gradient(
          cmap="YlGnBu", subset=pd.IndexSlice[tabla_result.index[:-1], :]
      )
      .set_table_styles([
          {
              "selector": "caption",
              "props": [
                  ("font-size", "16px"),
                  ("font-weight", "bold"),
                  ("padding", "10px"),
                  ("color", "#2c3e50"),
              ],
          },
          {
              "selector": "th",
              "props": [
                  ("background-color", "#2c3e50"),
                  ("color", "white"),
                  ("text-align", "center"),
                  ("padding", "10px"),
                  ("font-size", "13px"),
              ],
          },
          {
              "selector": "td",
              "props": [
                  ("text-align", "center"),
                  ("padding", "8px"),
                  ("font-size", "12px"),
              ],
          },
      ])
      .set_properties(**{"border": "1px solid #e0e0e0"})
  )

  display(estilo_result)

  # 3. Filtrar variables y guardar dataframes limpios
  dataframes_filtrados = {
      año: filtrar_variables(df, VARIABLES_UTILES)
      for año, df in modulos_1.items()
  }

  for año, df in dataframes_filtrados.items():
    ruta_salida = CARPETA_SALIDA / f"enaho_mod1_{año}_filtrado.csv"
    df.to_csv(ruta_salida, index=False, encoding="utf-8")
    print(f"Guardado: {ruta_salida}")

  return dataframes_filtrados


if __name__ == "__main__":
  dataframes_filtrados = main()

2019: 43868 filas, 323 columnas
2020: 53423 filas, 331 columnas
2021: 43524 filas, 331 columnas
2022: 44122 filas, 324 columnas
2023: 44378 filas, 338 columnas
2024: 44731 filas, 338 columnas
2025: 44599 filas, 336 columnas

Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\porcentaje_result_por_año.csv



,2019,2020,2021,2022,2023,2024,2025
result,,,,,,,
01. Completa,66.06%,58.38%,69.17%,66.21%,64.85%,61.75%,61.12%
02. Incompleta,12.73%,6.18%,9.51%,11.33%,11.51%,13.57%,14.44%
03. Rechazo,3.27%,2.44%,3.13%,3.07%,3.59%,3.66%,3.56%
04. Ausente,0.52%,0.83%,0.82%,0.80%,0.60%,0.55%,0.66%
05. Deshabitada,5.50%,3.47%,6.19%,6.42%,6.58%,7.02%,6.82%
07. Otro,11.92%,28.70%,11.18%,12.17%,12.87%,13.45%,13.40%
Total,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%


Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2019_filtrado.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2020_filtrado.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2021_filtrado.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2022_filtrado.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2023_filtrado.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2024_filtrado.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME\enaho_mod1_2025_filtrado.csv
